In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from datetime import datetime
from dateutil.relativedelta import relativedelta
import ast

In [0]:
dbutils.widgets.text('silver_params', '')

In [0]:
silver_params = dbutils.widgets.get('silver_params')

params = ast.literal_eval(silver_params)

table_name = params.get('table_name')
natural_keys = params.get('natural_keys')
data = (datetime.now()-relativedelta(months=1)).strftime('%Y%m01')
table_name, natural_keys, params, data

('transacoes',
 ['id', 'conta_id', '_businessdate'],
 {'table_name': 'transacoes',
  'natural_keys': ['id', 'conta_id', '_businessdate']},
 '20260601')

In [0]:
def process_silver():
    df_bronze = (spark.read.table(f'mobills.bronze.{table_name}'))
    max_ingesttime = df_bronze.select(max('_ingesttime')).collect()[0][0]
    df_bronze_new = df_bronze.filter(col('_ingesttime')==max_ingesttime)

    # generate hash primary key
    df_bronze_pk = (
        df_bronze_new
        .withColumn('_hash_pk', sha2(concat_ws('~', *natural_keys), 256))
    )

    # dedup
    w = (
        Window
        .partitionBy(natural_keys)
        .orderBy(col('_processdate').desc(), col('_ingesttime').desc())
    )
    df_dedup = (
        df_bronze_pk
        .withColumn('rowid', row_number().over(w))
        .filter(col('rowid') == 1)
        .drop('rowid')
    )

    # clear and reprocess future
    if not spark.catalog.tableExists(f'mobills.silver.{table_name}'):
        (
            df_dedup
            .write
            .partitionBy(['_processdate', '_businessdate'])
            .format('delta')
            .mode('append')
            .saveAsTable(f'mobills.silver.{table_name}')
        )
        return
    delta_silver = DeltaTable.forName(spark, f"mobills.silver.{table_name}")
    
    inicio_janela = data
    condicao_delete = f"_businessdate >= {inicio_janela}"
    delta_silver.delete(condition=condicao_delete)

    # write
    (
        df_dedup
        .write
        .partitionBy(['_processdate', '_businessdate'])
        .format('delta')
        .mode('append')
        .saveAsTable(f'mobills.silver.{table_name}')
    )

    return df_dedup

In [0]:
process_silver()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4726238223878673>, line 1
----> 1 process_silver()

File <command-7965958392920776>, line 39, in process_silver()
     30 delta_silver.delete(condition=condicao_delete)
     32 # write
     33 (
     34     df_dedup
     35     .write
     36     .partitionBy(['_processdate', '_businessdate'])
     37     .format('delta')
     38     .mode('append')
---> 39     .saveAsTable(f'mobills.silver.{table_name}')
     40 )
     42 return df_dedup

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.obs